In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Any, Dict
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client= TavilyClient()

db= SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def web_search(query: str) ->Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) ->str:
    """Execute a SQLite query against the Chinook database and return the result. You MUST use this tool to fetch counts or data."""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"    

C:\Users\Best By\AppData\Local\Temp\ipykernel_19464\2122497008.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class User_Role:
    user_role: str= "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: callable[[ModelRequest], ModelResponse]) ->ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role= request.runtime.context.user_role

    if user_role== "internal":
        pass
    else:
        tools=[web_search]
        request= request.override(tools= tools)
    return handler(request)    

In [5]:
from langchain.agents import create_agent

agent= create_agent(
    model= "gpt-5-nano",
    middleware=[dynamic_tool_call],
    tools= [web_search, sql_query],
    context_schema=User_Role
)

In [6]:
from langchain.messages import HumanMessage

response= agent.invoke(
    {
        "messages": [HumanMessage(content="How many artists are in the database?")]
    },
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t tell the exact number. Which database or dataset are you referring to? If you share the schema or how you access it, I can give you the exact command. In the meantime, here are common ways to count artists:

- SQL (generic)
  - Basic count: SELECT COUNT(*) AS total_artists FROM artists;
  - If the table is named artist: SELECT COUNT(*) FROM artist;
  - Excluding inactive artists: SELECT COUNT(*) AS total_artists FROM artists WHERE is_active = true;
  - Distinct by id (in case of duplicates): SELECT COUNT(DISTINCT artist_id) FROM artists;

- If you’re using an API
  - Many APIs paginate results and return total in metadata, e.g., GET /artists might return { total: 123, items: [...] }. Check the response for a total count or the X-Total-Count header.

- CSV/Excel
  - Open the file and count the rows excluding the header, or in Python: import pandas as pd; df = pd.read_csv('artists.csv'); len(df)

If you share the platform (MySQL, PostgreSQ

In [10]:
response= agent.invoke(
    {
        "messages": [HumanMessage(content="How many artists are in the database?")]
    },
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

There are 275 artists in the database (Artist table).


In [11]:
print(response)

{'messages': [HumanMessage(content='How many artists are in the database?', additional_kwargs={}, response_metadata={}, id='0b818365-a943-49c0-bc32-1bc82d7b1672'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 287, 'prompt_tokens': 171, 'total_tokens': 458, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrU0AF9crR7ax2eXJX1XBDTMiNcjc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed1f6-f2d9-74b1-b59e-ee15391c89ec-0', tool_calls=[{'name': 'sql_query', 'args': {'query': 'SELECT COUNT(*) AS ArtistCount FROM Artist;'}, 'id': 'call_XRq4i848bSAwF3gub0jh4mtx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [8]:
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [9]:
print(db.run("SELECT COUNT(*) FROM Artist"))

[(275,)]
